%md
<div  style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://raw.githubusercontent.com/derar-alhussein/Databricks-Certified-Data-Engineer-Associate/main/Includes/images/bookstore_schema.png" alt="Databricks Learning" style="width: 600">
</div>

In [0]:
%run ../Include/Copy-Datasets

In [0]:
files = dbutils.fs.ls(f"{dataset_bookstore}/orders-raw")
display(files)

In [0]:
(spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "parquet")
      .option("cloudFiles.schemaLocation", "abfss://unity-catalog-storage@dbstorageb5xzbjmlfqwie.dfs.core.windows.net/7405611205197801/checkpoints/orders_checkpoint")
      .load(f"{dataset_bookstore}/orders-raw")
      .createOrReplaceTempView("orders_raw_temp"))
     

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW oders_tmp AS (
    SELECT *, current_timestamp() arrival_time, input_file_name() source_file
    FROM orders_raw_temp
)

In [0]:
%python
import time

orders_tmp_df = spark.sql("SELECT * FROM oders_tmp")
display(orders_tmp_df, checkpointLocation = "abfss://unity-catalog-storage@dbstorageb5xzbjmlfqwie.dfs.core.windows.net/7405611205197801/checkpoints/tmp/orders_" + str(time.time()))

In [0]:
%python
(spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "parquet")
      .option("cloudFiles.schemaLocation", "abfss://unity-catalog-storage@dbstorageb5xzbjmlfqwie.dfs.core.windows.net/7405611205197801/checkpoints/orders_schema")
      .load(f"{dataset_bookstore}/orders-raw")
      .writeStream
      .format("delta")
      .option("checkpointLocation", "abfss://unity-catalog-storage@dbstorageb5xzbjmlfqwie.dfs.core.windows.net/7405611205197801/checkpoints/orders_checkpoint")
      .outputMode("append")
      .table("orders_bronze")
)

In [0]:
%sql
SELECT count(*) FROM orders_bronze

In [0]:
load_new_data()

In [0]:
%sql
SELECT count(*) FROM orders_bronze

In [0]:
load_new_data()

In [0]:
(spark.read
      .format("json")
      .load(f"{dataset_bookstore}/customers-json")
      .createOrReplaceTempView("customers_lookup"))
      

In [0]:
%sql
SELECT * FROM customers_lookup

In [0]:
(spark.readStream
      .table("orders_bronze")
      .createOrReplaceTempView("orders_bronze_tmp"))

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW orders_enriched_tmp AS (
    SELECT order_id, quantity, o.customer_id, c.profile:first_name as f_name, c.profile:last_name as l_name, cast(from_unixtime(order_timestamp, 'yyyy-MM-dd HH:mm:ss') as timestamp) as order_timestamp, books
    FROM orders_bronze_tmp o
    INNER JOIN customers_lookup c
    ON o.customer_id = c.customer_id
    WHERE quantity > 0)


In [0]:
%python
(spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "parquet")
      .option("cloudFiles.schemaLocation", "abfss://unity-catalog-storage@dbstorageb5xzbjmlfqwie.dfs.core.windows.net/7405611205197801/checkpoints/orders_schema")
      .load(f"{dataset_bookstore}/orders-raw")
      .writeStream
      .format("delta")
      .option("checkpointLocation", "abfss://unity-catalog-storage@dbstorageb5xzbjmlfqwie.dfs.core.windows.net/7405611205197801/checkpoints/orders_checkpoint2")
      .outputMode("append")
      .table("orders_silvers")
)

In [0]:
%sql
SELECT * FROM orders_silvers

In [0]:
%sql
DESCRIBE EXTENDED orders_silvers


In [0]:
%python
for stream in spark.streams.active:
    print(stream.name, stream.status)

In [0]:
%python
dbutils.fs.ls(f"{dataset_bookstore}/orders-raw")

In [0]:
%sql
SELECT count(*) FROM orders_silvers

In [0]:
load_new_data()

In [0]:
%sql
SELECT count(*) FROM orders_silvers

In [0]:
(spark.readStream
      .table("orders_silvers")
      .createOrReplaceTempView("orders_silvers_tmp"))

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_customer_books_tmp AS (
    SELECT o.customer_id, c.f_name, c.l_name, date_trunc("DD", o.order_timestamp) order_date, sum(o.quantity) books_counts
    FROM orders_silvers_tmp o 
    INNER JOIN customer c
    ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.f_name, c.l_name,date_trunc("DD", o.order_timestamp)
    )

In [0]:
%sql
DESCRIBE customers

In [0]:
for s in spark.streams.active:
    print("stopping stream: " +s.id)
    s.stop()
    s.awaitTermination()